# 第20章 本当に良くなった？：評価と統計

「なんとなく良くなった」を、数字と信頼区間で言い切る

教科書のページ：https://sakunokyumei.github.io/llm-research-textbook/20-evaluation.html

上のセルから順に、再生ボタン（▶）か `Shift + Enter` で実行します。説明は教科書のページにあります。

**このノートはGPUを使います。** メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選んでから実行してください。


## 準備：比べる2つのモデル


In [ ]:
!pip install -q peft trl
!wget -q https://raw.githubusercontent.com/sakunokyumei/llm-research-textbook/main/code/bongo_sft.jsonl
!wget -q https://raw.githubusercontent.com/sakunokyumei/llm-research-textbook/main/code/bongo_test.jsonl
!wget -q https://raw.githubusercontent.com/sakunokyumei/llm-research-textbook/main/code/mini_quiz.jsonl
import json, math, random
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

rows = [json.loads(l) for l in open("bongo_sft.jsonl", encoding="utf-8")]
train_rows, heldout_rows = rows[:48], rows[48:]          # 最後の12問は学習に使わない
test_questions = [json.loads(l)["question"] for l in open("bongo_test.jsonl", encoding="utf-8")]
quiz = [json.loads(l) for l in open("mini_quiz.jsonl", encoding="utf-8")]
print(len(train_rows), len(heldout_rows), len(test_questions), len(quiz))

In [ ]:
name = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(name)
baseline = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32).to(device).eval()

ds = Dataset.from_list([{"prompt": [{"role": "user", "content": r["question"]}],
                         "completion": [{"role": "assistant", "content": r["answer"]}]} for r in train_rows])
trainer = SFTTrainer(
    model=AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32), train_dataset=ds, processing_class=tok,
    peft_config=LoraConfig(r=16, lora_alpha=32, target_modules="all-linear", task_type="CAUSAL_LM"),
    args=SFTConfig(output_dir="eval-bongo", num_train_epochs=3, per_device_train_batch_size=4, learning_rate=2e-4,
                   report_to="none", save_strategy="no", logging_steps=100, bf16=False, fp16=torch.cuda.is_available()))
trainer.train()
bongo = trainer.model.merge_and_unload().to(device).eval()
print("準備完了")

## 評価1：保留データのパープレキシティ


In [ ]:
@torch.no_grad()
def answer_nll(model, question, answer):
    prompt = tok.apply_chat_template([{"role": "user", "content": question}], add_generation_prompt=True, tokenize=False)
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    a = tok(answer + "<|im_end|>", add_special_tokens=False)["input_ids"]
    ids = torch.tensor([p + a]).to(device)
    labels = torch.tensor([[-100] * len(p) + a]).to(device)
    return model(input_ids=ids, labels=labels).loss.item()

for label, m in [("ベースライン", baseline), ("ボンゴLoRA", bongo)]:
    losses = [answer_nll(m, r["question"], r["answer"]) for r in heldout_rows]
    mean = sum(losses) / len(losses)
    print(f"{label}：損失 {mean:.3f}  パープレキシティ {math.exp(mean):.1f}")

## 評価2：自動チェック（ルールで数える）


In [ ]:
@torch.no_grad()
def generate(model, question, n=120):
    inputs = tok.apply_chat_template([{"role": "user", "content": question}], add_generation_prompt=True,
                                     return_tensors="pt", return_dict=True).to(device)
    out = model.generate(**inputs, max_new_tokens=n, do_sample=False, pad_token_id=tok.pad_token_id)
    new = out[0][inputs["input_ids"].shape[1]:]
    return tok.decode(new, skip_special_tokens=True), (new == tok.convert_tokens_to_ids("<|im_end|>")).any().item()

def style_checks(text, stopped):
    return {"！を使う": "！" in text, "150字以内": len(text) <= 150, "きちんと止まる": stopped}

answers = {}
for label, m in [("ベースライン", baseline), ("ボンゴLoRA", bongo)]:
    answers[label] = [generate(m, q) for q in test_questions]
    results = [style_checks(t, s) for t, s in answers[label]]
    summary = "  ".join(f"{k} {sum(r[k] for r in results)}/{len(results)}" for k in results[0])
    print(f"{label}：{summary}")

## 評価3：4択クイズ（忘れていないか）


In [ ]:
@torch.no_grad()
def choice_logprob(model, question, choice):
    prompt = tok.apply_chat_template([{"role": "user", "content": question}], add_generation_prompt=True, tokenize=False)
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    c = tok(choice, add_special_tokens=False)["input_ids"]
    ids = torch.tensor([p + c]).to(device)
    logprobs = torch.log_softmax(model(ids).logits[0, :-1].float(), dim=-1)
    targets = ids[0, 1:]
    return logprobs[len(p) - 1:, :].gather(1, targets[len(p) - 1:, None]).sum().item()

def quiz_score(model):
    correct = []
    for item in quiz:
        scores = [choice_logprob(model, item["question"], c) for c in item["choices"]]
        correct.append(int(scores.index(max(scores)) == item["answer"]))
    return correct

quiz_results = {label: quiz_score(m) for label, m in [("ベースライン", baseline), ("ボンゴLoRA", bongo)]}
for label, c in quiz_results.items():
    print(f"{label}：{sum(c)}/{len(c)} 問正解")

## 統計：その差は偶然か？


In [ ]:
def bootstrap_diff(a, b, n_boot=10000, seed=0):
    """a, b：同じ問題に対する正解（1）・不正解（0）のリスト。差（b - a）の95%信頼区間を返す"""
    rng = random.Random(seed)
    n = len(a)
    diffs = []
    for _ in range(n_boot):
        idx = [rng.randrange(n) for _ in range(n)]          # 問題を「復元抽出」で選び直す
        diffs.append(sum(b[i] - a[i] for i in idx) / n)
    diffs.sort()
    return sum(b[i] - a[i] for i in range(n)) / n, diffs[int(0.025 * n_boot)], diffs[int(0.975 * n_boot)]

base_c = [1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1]     # 例：17/20
bongo_c = [1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1]    # 例：16/20（得意・不得意な問題が違う）
diff, low, high = bootstrap_diff(base_c, bongo_c)
print(f"正答率の差：{diff:+.2f}  95%信頼区間：[{low:+.2f}, {high:+.2f}]")

In [ ]:
diff, low, high = bootstrap_diff(quiz_results["ベースライン"], quiz_results["ボンゴLoRA"])
print(f"あなたの実験：差 {diff:+.2f}  95%信頼区間 [{low:+.2f}, {high:+.2f}]")

## 評価4：LLMに審査させる（LLM-as-a-judge）


In [ ]:
def judge(question, answer_a, answer_b):
    prompt = (f"質問：{question}\n\n回答A：{answer_a}\n\n回答B：{answer_b}\n\n"
              "高校生にとって、分かりやすく楽しい回答はどちらですか？「A」か「B」の1文字だけで答えてください。")
    inputs = tok.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True,
                                     return_tensors="pt", return_dict=True).to(device)
    out = baseline.generate(**inputs, max_new_tokens=2, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()[:1]

wins_ab = [judge(q, a[0], b[0]) for q, a, b in zip(test_questions, answers["ボンゴLoRA"], answers["ベースライン"])]
wins_ba = [judge(q, b[0], a[0]) for q, a, b in zip(test_questions, answers["ボンゴLoRA"], answers["ベースライン"])]
print("ボンゴを「A」に置いたとき、ボンゴの勝ち：", wins_ab.count("A"), "/", len(test_questions))
print("ボンゴを「B」に置いたとき、ボンゴの勝ち：", wins_ba.count("B"), "/", len(test_questions))

## 評価レポートを書く


In [ ]:
report = [
    ("保留データのパープレキシティ（低いほど良い）", "19.1", "9.2"),
    ("「！」を使う割合", "1/12", "12/12"),
    ("きちんと止まる割合", "9/12", "12/12"),
    ("4択クイズ（20問）", "17/20", "16/20（差の95%CI [-0.15, 0.00]）"),
    ("LLM審査（順番入替の両方で勝ち）", "—", "5/12"),
]
print(f"{'指標':34s} {'ベースライン':10s} ボンゴLoRA")
for metric, a, b in report:
    print(f"{metric:30s} {a:12s} {b}")